In [510]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [511]:
data_path = "../data/"

df_c = pd.read_csv(data_path + "claims.csv")
df_cp = pd.read_csv(data_path + "claim_payments.csv")
df_ibf = pd.read_csv(data_path + "industry_benchmark_factors.csv")

In [521]:
# Join df_c and df_cp together on claim_id
df = df_c.merge(df_cp, on='claim_id', how='right')

### Ultimate Loss Calculator

Cumulative paid claims loss development triangle:

In [ ]:
# Define a helper function producing a pivot table of payments by payment year and accident year
def aggregate_payments(ctype: str) -> pd.DataFrame:
    # Check if input is correct
    if ctype not in df_c['claim_type'].unique().tolist():
        raise Exception(f'You need to input a claim type (as a string) from: {df_c['claim_type'].unique().tolist()}')

    # Create the aggregate payments data frame by payment year and accident year
    df_ap = pd.pivot_table(df[df['claim_type'] == ctype], 
                            values="payment_amount",index="claim_id", 
                            columns="payment_year", 
                            aggfunc="sum")
    df_ap = df_ap.fillna(0)
    df_ap['total_claim_payment'] = df_ap.sum(axis=1)
    df_ap = df_ap.merge(df_c[['claim_id', 'accident_year']], 
                        on="claim_id", 
                        how="left")

    return df_ap

# Define a function showing the LDT for a certain claim type.
def ldt(ctype: str) -> pd.DataFrame : 
    # Check if input is correct
    if ctype not in df_c['claim_type'].unique().tolist():
        raise Exception(f'You need to input a claim type (as a string) from: {df_c['claim_type'].unique().tolist()}')

    # Create the aggregate payments data frame by payment year and accident year
    df_ap = aggregate_payments(ctype)
    
    # Create the columns of the LDT
    cols = ['accident_year'] + [('developmental_period', f'{12*i}->{12*(i+1)}') for i in range(8)]

    # Create the LDT
    ldt = []
    for j in range(8):
        l = [str(2018 + j)]
        for i in range(8 - j):
            pt = df_ap[(df_ap['accident_year'] == 2018 + j) & (df_ap['total_claim_payment'] < 250000)]
            val = pt[range(2018 + j,2018 + i + j + 1)].to_numpy().sum()
            l.append(np.round(val, 2))
        if len(l) - 1 < 8:
            l += [np.nan for _ in range(9 - len(l))]
        ldt.append(l)

    ldt = pd.DataFrame(np.array(ldt), columns=cols)
    return ldt.apply(pd.to_numeric, errors="coerce")

In [515]:
df_ldt = ldt('Water Damage')
df_ldt

,accident_year,"(developmental_period, 0->12)","(developmental_period, 12->24)","(developmental_period, 24->36)","(developmental_period, 36->48)","(developmental_period, 48->60)","(developmental_period, 60->72)","(developmental_period, 72->84)","(developmental_period, 84->96)"
0,2018,18750.69,35749.99,35749.99,35749.99,35749.99,35749.99,35749.99,35749.99
1,2019,61545.59,95625.68,95625.68,95625.68,95625.68,95625.68,95625.68,NaN
2,2020,64692.74,88920.97,88920.97,88920.97,88920.97,88920.97,NaN,NaN
3,2021,72080.05,132370.56,134172.15,134172.15,134172.15,NaN,NaN,NaN
4,2022,71267.80,122293.96,122293.96,122293.96,NaN,NaN,NaN,NaN
5,2023,110404.26,163538.19,165293.16,NaN,NaN,NaN,NaN,NaN
6,2024,112595.13,182005.26,NaN,NaN,NaN,NaN,NaN,NaN
7,2025,86727.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Average loss developmental factors:

In [482]:
def average_ldfs(ldt: pd.DataFrame, n: int) -> pd.DataFrame:
    '''
        ldt: Loss development triangle (as a DataFrame object)
        n: the number of additonal columns to append
    '''
    if n < 0:
        raise Exception('n must be a nonnegative integer')
    
    numrows = ldt.shape[0]
    numcols = ldt.shape[1] - 1
    cols = ldt.columns[2:].to_list()
    ldfs = []

    # Compute average of pairwise factors
    for i in range(numrows - 1):
        pair = ldt.iloc[0:numrows - 1 - i, i + 1: i + 3]
        pair['factor'] = pair.iloc[:, 1] / pair.iloc[:, 0]
        ldfs.append(pair['factor'].mean(numeric_only=True))

    # Pad extra columns if needed
    if n == 0:
        cols[-1] = ('developmental_period', cols[-1][1][0 : cols[-1][1].find('>') + 1] + 'Ult')
    if n > 0:
        cols += [('developmental_period', f'{12 * (numcols + i)}->{12 * (numcols + i + 1)}') for i in range(n)]
        cols[-1] = ('developmental_period', cols[-1][1][0 : cols[-1][1].find('>') + 1] + 'Ult')
        ldfs += [ float(1) for _ in range(n)]

    return pd.DataFrame([ldfs], columns=cols)

In [496]:
average_ldfs_list = []
for claim_type in df_c['claim_type'].unique().tolist():
    avg_ldfs = average_ldfs(ldt(claim_type), 3)
    avg_ldfs.insert(0, 'claim_type', claim_type)
    average_ldfs_list.append(avg_ldfs)
pd.concat(average_ldfs_list, axis=0)

,claim_type,"(developmental_period, 12->24)","(developmental_period, 24->36)","(developmental_period, 36->48)","(developmental_period, 48->60)","(developmental_period, 60->72)","(developmental_period, 72->84)","(developmental_period, 84->96)","(developmental_period, 96->108)","(developmental_period, 108->120)","(developmental_period, 120->Ult)"
0,Wind/Hail,1.160758,1.000000,1.00000,1.0,1.0,1.0,1.0,1.0,1.0,1.0
0,Water Damage,1.640712,1.004057,1.00000,1.0,1.0,1.0,1.0,1.0,1.0,1.0
0,Fire,2.775173,1.188071,1.00137,1.0,1.0,1.0,1.0,1.0,1.0,1.0
0,Flood,2.617605,1.041029,1.00000,1.0,1.0,1.0,1.0,1.0,1.0,1.0
0,Theft,1.115912,1.000000,1.00000,1.0,1.0,1.0,1.0,1.0,1.0,1.0


Credibility factor:

In [516]:
def credf(ctype: str) -> float: 
    # Check if input is correct
    if ctype not in df_c['claim_type'].unique().tolist():
        raise Exception(f'You need to input a claim type (as a string) from: {df_c['claim_type'].unique().tolist()}')

    df_ap = aggregate_payments(ctype)

    return df_ap.shape[0] / 1082.0

In [517]:
credf('Wind/Hail')

0.10166358595194085

In [ ]:
aggregate_payments('Wind/Hail')
# Note: It's missing data with claim_id 10481

,claim_id,2018,2019,2020,2021,2022,2023,2024,2025,total_claim_payment,accident_year
0,10001,4777.43,0.00,0.00,0.00,0.0,0.00,0.0,0.00,4777.43,2018
1,10003,0.00,0.00,0.00,0.00,0.0,4640.37,0.0,0.00,4640.37,2023
2,10004,0.00,0.00,0.00,0.00,0.0,0.00,0.0,6517.61,6517.61,2024
3,10015,4333.54,0.00,0.00,0.00,0.0,0.00,0.0,0.00,4333.54,2018
4,10020,0.00,8405.94,0.00,0.00,0.0,0.00,0.0,0.00,8405.94,2019
...,...,...,...,...,...,...,...,...,...,...,...
105,10479,0.00,0.00,0.00,1721.76,0.0,0.00,0.0,0.00,1721.76,2020
106,10480,0.00,0.00,0.00,0.00,7906.2,830.27,0.0,0.00,8736.47,2022
107,10486,0.00,0.00,4811.09,0.00,0.0,0.00,0.0,0.00,4811.09,2020
108,10488,0.00,13428.56,0.00,0.00,0.0,0.00,0.0,0.00,13428.56,2019


In [568]:
aggregate_payments('Water Damage')
# It's missing data claim_id 10475

,claim_id,2018,2019,2020,2021,2022,2023,2024,2025,total_claim_payment,accident_year
0,10002,0.00,0.00,0.00,0.0,5454.30,881.28,0.00,0.00,6335.58,2022
1,10006,0.00,0.00,0.00,0.0,0.00,0.00,4538.25,0.00,4538.25,2024
2,10007,2435.31,2449.96,0.00,0.0,0.00,0.00,0.00,0.00,4885.27,2018
3,10009,0.00,553.12,1745.98,0.0,0.00,0.00,0.00,0.00,2299.10,2019
4,10013,0.00,0.00,0.00,0.0,1450.68,1600.44,0.00,0.00,3051.12,2022
...,...,...,...,...,...,...,...,...,...,...,...
148,10477,0.00,0.00,0.00,0.0,0.00,0.00,1052.95,1002.89,2055.84,2024
149,10482,0.00,0.00,0.00,0.0,0.00,2917.19,2187.03,0.00,5104.22,2023
150,10484,0.00,0.00,0.00,0.0,0.00,7769.55,0.00,0.00,7769.55,2023
151,10485,0.00,2755.71,314.54,0.0,0.00,0.00,0.00,0.00,3070.25,2019


In [559]:
df_ap = pd.pivot_table(df[df['claim_type'] == 'Water Damage'], 
                            values="payment_amount",index="claim_id", 
                            columns="payment_year", 
                            aggfunc="sum")
df_ap

payment_year,2018,2019,2020,2021,2022,2023,2024,2025
claim_id,,,,,,,,
10002,NaN,NaN,NaN,NaN,5454.30,881.28,NaN,NaN
10006,NaN,NaN,NaN,NaN,NaN,NaN,4538.25,NaN
10007,2435.31,2449.96,NaN,NaN,NaN,NaN,NaN,NaN
10009,NaN,553.12,1745.98,NaN,NaN,NaN,NaN,NaN
10013,NaN,NaN,NaN,NaN,1450.68,1600.44,NaN,NaN
...,...,...,...,...,...,...,...,...
10477,NaN,NaN,NaN,NaN,NaN,NaN,1052.95,1002.89
10482,NaN,NaN,NaN,NaN,NaN,2917.19,2187.03,NaN
10484,NaN,NaN,NaN,NaN,NaN,7769.55,NaN,NaN


In [ ]:
df_c[ df_c['claim_type'] == 'Wind/Hail' ][-25: -1]

,claim_id,claim_type,accident_year,accident_month,status
396,10397,Wind/Hail,2024,8,Open
397,10398,Wind/Hail,2020,3,Closed
399,10400,Wind/Hail,2023,9,Open
406,10407,Wind/Hail,2023,10,Open
407,10408,Wind/Hail,2025,10,Open
416,10417,Wind/Hail,2020,6,Closed
424,10425,Wind/Hail,2024,5,Open
427,10428,Wind/Hail,2023,10,Closed
431,10432,Wind/Hail,2018,2,Closed
435,10436,Wind/Hail,2019,12,Closed


In [ ]:
df_cp[-75: -25]

,payment_id,claim_id,payment_amount,payment_year,payment_month
2343,22344,10473,1163.39,2023,2
2344,22345,10473,445.05,2023,9
2345,22346,10474,14566.35,2022,10
2346,22347,10474,2163.35,2023,5
2347,22348,10476,5132.19,2023,6
2348,22349,10477,559.38,2024,10
2349,22350,10477,493.57,2024,12
2350,22351,10477,699.53,2025,5
2351,22352,10477,303.36,2025,7
2352,22353,10478,7536.83,2024,8


In [536]:
df[ df['claim_id'] == 10481]

,claim_id,claim_type,accident_year,accident_month,status,payment_id,payment_amount,payment_year,payment_month


In [561]:
df = df_c.merge(df_cp, on='claim_id', how='right')
df[ df['claim_type'] == 'Wind/Hail'][-25: -1]

,claim_id,claim_type,accident_year,accident_month,status,payment_id,payment_amount,payment_year,payment_month
2143,10425,Wind/Hail,2024,5,Open,22144,2702.05,2025,1
2148,10428,Wind/Hail,2023,10,Closed,22149,4598.70,2024,1
2174,10432,Wind/Hail,2018,2,Closed,22175,4988.00,2018,4
2184,10436,Wind/Hail,2019,12,Closed,22185,5751.26,2020,1
2185,10436,Wind/Hail,2019,12,Closed,22186,358.77,2020,8
2190,10438,Wind/Hail,2025,5,Open,22191,4308.73,2025,8
2191,10438,Wind/Hail,2025,5,Open,22192,1253.93,2025,12
2193,10440,Wind/Hail,2020,9,Closed,22194,3395.93,2020,11
2198,10442,Wind/Hail,2023,10,Open,22199,7016.79,2024,1
2249,10448,Wind/Hail,2025,1,Open,22250,9193.79,2025,3


In [566]:
df[ df['claim_type'] == 'Water Damage'][-25: -1]

,claim_id,claim_type,accident_year,accident_month,status,payment_id,payment_amount,payment_year,payment_month
2330,10470,Water Damage,2025,11,Open,22331,354.83,2025,11
2341,10473,Water Damage,2022,9,Closed,22342,442.34,2022,10
2342,10473,Water Damage,2022,9,Closed,22343,743.87,2022,11
2343,10473,Water Damage,2022,9,Closed,22344,1163.39,2023,2
2344,10473,Water Damage,2022,9,Closed,22345,445.05,2023,9
2348,10477,Water Damage,2024,10,Open,22349,559.38,2024,10
2349,10477,Water Damage,2024,10,Open,22350,493.57,2024,12
2350,10477,Water Damage,2024,10,Open,22351,699.53,2025,5
2351,10477,Water Damage,2024,10,Open,22352,303.36,2025,7
2358,10482,Water Damage,2023,7,Closed,22359,1187.99,2023,7


In [557]:
# Left DataFrame
left = pd.DataFrame({
    "id": [1, 2, 3],
    "age": [25, 30, 40]
})

# Right DataFrame
right = pd.DataFrame({
    "id": [1, 3],
    "claims": [3, 2]
})

# Left join
result = left.merge(
    right,
    on="id",
    how="left"
)

left, right, result

(   id  age
 0   1   25
 1   2   30
 2   3   40,
    id  claims
 0   1       3
 1   3       2,
    id  age  claims
 0   1   25     3.0
 1   2   30     NaN
 2   3   40     2.0)